In [10]:
import numpy as np
import pandas as pd
import torch
import ast

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

DATA_PATH        = "../Test Dataset Creation/final_BERT_labeled_reddit_comments.csv"
PRETRAINED_MODEL = "../.saved_models/FinalSentimentModel"
MODEL_SAVE       = "../.saved_models/ScrapedSentimentModel"
CKPT_DIR         = "/tmp/scraped_checkpoints"

In [11]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "datasets", "accelerate", "scikit-learn"])

CompletedProcess(args=['/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/bin/python', '-m', 'pip', 'install', '-q', 'transformers', 'datasets', 'accelerate', 'scikit-learn'], returncode=0)

In [12]:
# GoEmotions label set — order matches FinalSentimentModel to keep the pipeline consistent
label_names = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]
num_labels = len(label_names)
label2id = {label: i for i, label in enumerate(label_names)}

print("Number of labels:", num_labels)
print("Labels:", label_names)

Number of labels: 28
Labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [13]:
# Load and parse the scraped Reddit dataset
df = pd.read_csv(DATA_PATH)

# Drop rows with missing text or emotion
df = df.dropna(subset=["comment_body", "predicted_emotion"]).reset_index(drop=True)

# Combine post title + comment body to match inference pipeline
df["combined_text"] = df["post_title"].fillna("") + " " + df["comment_body"].fillna("")

def parse_labels(label_str):
    """Convert string-list like "['anger', 'disgust']" to list of label ids."""
    try:
        labels = ast.literal_eval(label_str)
        return [label2id[l] for l in labels if l in label2id]
    except Exception:
        return []

df["label_ids"] = df["predicted_emotion"].apply(parse_labels)

# Drop rows where parsing yielded no valid labels
df = df[df["label_ids"].map(len) > 0].reset_index(drop=True)

print(f"Dataset size after cleaning: {len(df)} rows")
print(df[["combined_text", "predicted_emotion", "label_ids"]].head(3))

Dataset size after cleaning: 1841 rows
                                       combined_text  \
0      Serious question mutually assured destruction   
1  Serious question Please- he will burn down the...   
2  Serious question I really don’t think he cares...   

          predicted_emotion label_ids  
0               ['neutral']      [27]  
1      ['disgust', 'anger']   [11, 2]  
2  ['disapproval', 'anger']   [10, 2]  


In [14]:
# Train / validation split (90/10)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 1656 | Val: 185


In [15]:
# Load tokenizer from FinalSentimentModel to stay consistent with the base model
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)

def encode_split(dataframe):
    texts = dataframe["combined_text"].tolist()
    label_ids_list = dataframe["label_ids"].tolist()

    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    labels = []
    for ids in label_ids_list:
        encoded = np.zeros(num_labels, dtype=np.float32)
        encoded[ids] = 1.0
        labels.append(encoded)

    tokenized["labels"] = labels
    return tokenized

def make_dataset(dataframe):
    encoded = encode_split(dataframe)
    ds = Dataset.from_dict(encoded)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)

print("Train dataset:", train_dataset)
print("Val dataset:", val_dataset)

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1656
})
Val dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 185
})


In [16]:
# Compute class-balanced positive weights for BCEWithLogitsLoss
labels_matrix = np.array([row.numpy() for row in train_dataset["labels"]])

pos_counts = labels_matrix.sum(axis=0)
neg_counts = labels_matrix.shape[0] - pos_counts

# Dampened weighting using square root (same as GoEmotionsImprovedRoBERTa)
pos_weight_calc = np.sqrt(neg_counts / (pos_counts + 1e-6))
pos_weight = torch.tensor(pos_weight_calc, dtype=torch.float)

print("Sample weights (first 5):", pos_weight[:5])

Sample weights (first 5): tensor([10.1242,  3.0061,  1.8676,  2.0552,  2.6679])


In [17]:
# Load from FinalSentimentModel (GoEmotions fine-tuned) for domain adaptation
model = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED_MODEL,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
    }

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2541.05it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


In [18]:
use_cuda = torch.cuda.is_available()
use_mps  = torch.backends.mps.is_available()

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=use_cuda,
    report_to="none"
)

print(f"device: {'cuda' if use_cuda else 'mps' if use_mps else 'cpu'}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


device: mps


In [19]:
class WeightedTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device)
        )

        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    pos_weight=pos_weight,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [20]:
trainer.train()

# Save the best model (loaded back automatically) to Drive
trainer.save_model(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)
print(f"Model saved to {MODEL_SAVE}")

/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Macro Precision,Macro Recall
1,0.342503,0.302544,0.244613,0.431444,0.291499,0.259586
2,0.308918,0.288063,0.339325,0.492091,0.374054,0.347439
3,0.270010,0.285521,0.334237,0.500835,0.353464,0.343775


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.69it/s]
/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]
/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.54it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.

Model saved to ../.saved_models/ScrapedSentimentModel


In [21]:
# ── Per-label threshold optimization ──────────────────────────────────────────
# Get validation logits from the best model
val_output = trainer.predict(val_dataset)
val_logits = val_output.predictions
if isinstance(val_logits, tuple):
    val_logits = val_logits[0]

val_probs = 1 / (1 + np.exp(-val_logits))                  # (N, 28)
val_labels = np.array([row.numpy() for row in val_dataset["labels"]])  # (N, 28)

# Sweep thresholds [0.1, 0.95] and pick the one that maximises per-label F1
thresholds_to_try = np.arange(0.1, 0.96, 0.05)
best_thresholds = np.full(num_labels, 0.5)

for i in range(num_labels):
    best_f1 = -1
    for t in thresholds_to_try:
        preds_i = (val_probs[:, i] >= t).astype(int)
        f1 = f1_score(val_labels[:, i], preds_i, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresholds[i] = t

print("Optimized thresholds per label:")
for name, t in zip(label_names, best_thresholds):
    print(f"  {name:<20} {t:.2f}")

/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Optimized thresholds per label:
  admiration           0.50
  amusement            0.40
  anger                0.30
  annoyance            0.30
  approval             0.30
  caring               0.10
  confusion            0.50
  curiosity            0.50
  desire               0.50
  disappointment       0.55
  disapproval          0.20
  disgust              0.50
  embarrassment        0.10
  excitement           0.10
  fear                 0.45
  gratitude            0.55
  grief                0.10
  joy                  0.60
  love                 0.10
  nervousness          0.65
  optimism             0.25
  pride                0.10
  realization          0.30
  relief               0.25
  remorse              0.10
  sadness              0.60
  surprise             0.35
  neutral              0.40


In [22]:
# ── Re-evaluate with optimized thresholds ─────────────────────────────────────
opt_preds = (val_probs >= best_thresholds).astype(int)

print("=== Validation metrics with optimized thresholds ===")
print(f"  macro_f1:        {f1_score(val_labels, opt_preds, average='macro',     zero_division=0):.4f}")
print(f"  micro_f1:        {f1_score(val_labels, opt_preds, average='micro',     zero_division=0):.4f}")
print(f"  macro_precision: {precision_score(val_labels, opt_preds, average='macro', zero_division=0):.4f}")
print(f"  macro_recall:    {recall_score(val_labels, opt_preds, average='macro',    zero_division=0):.4f}")

import json
thresholds_path = f"{MODEL_SAVE}/optimal_thresholds.json"
with open(thresholds_path, "w") as f:
    json.dump(dict(zip(label_names, best_thresholds.tolist())), f, indent=2)
print(f"Thresholds saved to {thresholds_path}")

=== Validation metrics with optimized thresholds ===
  macro_f1:        0.3997
  micro_f1:        0.5216
  macro_precision: 0.3871
  macro_recall:    0.5190
Thresholds saved to ../.saved_models/ScrapedSentimentModel/optimal_thresholds.json


In [23]:
def predict_emotions(text, thresholds=None):
    """Predict emotions using per-label optimized thresholds (falls back to 0.5)."""
    if thresholds is None:
        thresholds = best_thresholds

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits).cpu().numpy()[0]

    return {
        label_names[i]: round(float(probs[i]), 4)
        for i in range(num_labels)
        if probs[i] >= thresholds[i]
    }

test_text = "I really don't think he cares about anyone but himself."
print(f"Test: '{test_text}'")
print(predict_emotions(test_text))

Test: 'I really don't think he cares about anyone but himself.'
{'anger': 0.3045, 'annoyance': 0.4452, 'disapproval': 0.6245}
